## This notebook requires GPU

This lab must be run in Google Colab in order to use GPU acceleration for model training. Click the button below to open this notebook in Colab, then set your runtime to GPU:

**Runtime > Change Runtime Type > T4 GPU**

### Upload the data files first

Before opening this notebook in Colab, be sure to download the data files from the course assets and upload them to a folder called `coursera-msds` in your Google Drive.

You will need:

    @verizon.zip
    @TMobile.zip
    @ATT.zip
    mobile_sentiment.csv
    mobile_sentiment_positive.csv
    mobile_sentiment_negative.csv


### Open in Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/msds-marketing-analytics/colab-notebooks/blob/main/NetworkAnalysis/MSDSNetworkAnalysis_Lesson_SemanticNetwork.ipynb)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
print("✅ Google Drive mounted successfully!")

# 🧠 Semantic Word Networks Analysis

## Verify MSDS kernel selection

The MSDS kernel is pre-installed with the libraries required by this notebook. If the kernel selection in the upper right of the Jupyter interface shows a different kernel, navigate to `Kernel > Change kernel > MSDS` before continuing.

![Screenshot%202026-03-07%20at%202.36.34%E2%80%AFPM.png](attachment:Screenshot%202026-03-07%20at%202.36.34%E2%80%AFPM.png)

## State-of-the-Art Semantic Network Construction from Twitter Data

### 🎯 **Objective**
This notebook implements cutting-edge semantic network analysis techniques to extract meaningful word relationships, semantic communities, and brand positioning insights from real Twitter data.

### 🔬 **Advanced Features**
- **Multi-scale semantic analysis** (word-level, phrase-level, sentence-level)
- **Transformer-based embeddings** (sentence-transformers)
- **PMI-based semantic similarity** with statistical significance testing
- **Advanced NLP preprocessing** (spaCy transformer pipeline)
- **Semantic community detection** with HDBSCAN + validation
- **Brand semantic positioning** analysis
- **Statistical validation framework** for all semantic relationships

### 📊 **Analysis Pipeline**
1. **Advanced text preprocessing** with NER and lemmatization
2. **Multi-scale semantic network construction**
3. **Contextual embedding generation**
4. **Statistical significance testing** for semantic relationships
5. **Semantic community detection** and validation
6. **Brand differentiation analysis** in semantic space
7. **Temporal semantic evolution** tracking

---

In [ ]:
from pathlib import Path
DATA_DIR = Path("/content/drive/MyDrive/coursera-msds")
DATA_DIR.mkdir(exist_ok=True)

## Data Setup Instructions

### the data directory Path
Ensure your data folder is located at:
```
/content/drive/MyDrive/coursera-msds
```

### Required Files
- `@ATT.zip` - AT&T Twitter data (10,000 tweets)
- `@TMobile.zip` - T-Mobile Twitter data (10,000 tweets)
- `@verizon.zip` - Verizon Twitter data (10,000 tweets)
- `mobile_sentiment.csv` - Sentiment analysis results
- `mobile_sentiment_positive.csv` - Positive sentiment tweets
- `mobile_sentiment_negative.csv` - Negative sentiment tweets

### Total Dataset
- **30,000 individual tweet JSON files** across 3 ZIP archives
- **Comprehensive sentiment labeling** for semantic-sentiment integration
- **Multi-brand comparative analysis** capability

---

In [ ]:
# May be needed for some jupyter environments

#import plotly.io as pio
#pio.renderers.default = "notebook_connected"

In [ ]:
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# Advanced NLP and Text Processing
import spacy
import nltk
from collections import Counter, defaultdict
from itertools import combinations, permutations
import re
import string

# Machine Learning and Embeddings
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.cluster import DBSCAN, KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sentence_transformers import SentenceTransformer
import umap
import hdbscan

# Statistical Analysis
from scipy import stats
from scipy.sparse import csr_matrix
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests

# Network Analysis
import community.community_louvain as community_louvain
try:
    import leidenalg
    import igraph as ig
    HAS_LEIDEN = True
except ImportError:
    HAS_LEIDEN = False
    print("⚠️ Leiden algorithm not available. Using Louvain instead.")

# Utilities
import json
import zipfile
import logging
import warnings
from pathlib import Path
from typing import List, Dict, Any, Tuple, Optional, Union
from dataclasses import dataclass, field
from tqdm.auto import tqdm
import time
import pickle

# Configure warnings and display
warnings.filterwarnings("ignore", category=UserWarning)
plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger("SemanticNetworkAnalysis")

print("✅ All imports loaded successfully!")
print(f"📊 NetworkX version: {nx.__version__}")
print(f"🔢 NumPy version: {np.__version__}")
print(f"📈 Pandas version: {pd.__version__}")
print(f"🧠 Leiden available: {HAS_LEIDEN}")


In [ ]:
@dataclass
class SemanticNetworkConfig:
    """Configuration for semantic network analysis."""

    # Data Processing
    min_word_frequency: int = 10  # Minimum word frequency for inclusion
    max_vocab_size: int = 10000   # Maximum vocabulary size
    min_tweet_length: int = 10    # Minimum tweet length (characters)
    max_tweet_length: int = 280   # Maximum tweet length (characters)

    # NLP Processing
    spacy_model: str = "en_core_web_sm"  # spaCy model for NLP
    use_lemmatization: bool = True        # Use lemmatization
    use_ner: bool = True                  # Use named entity recognition
    pos_tags_to_keep: List[str] = field(default_factory=lambda: ["NOUN", "ADJ", "VERB", "ADV"])

    # Note: To streamline this exercise and to make this notebook compatible with the Coursera
    # lab environment, the model is saved locally. For a non-local model, remove the leading
    # path from the model name and specify `local_files_only = False`

    # Semantic Similarity
    embedding_model: str = "all-mpnet-base-v2"  # Sentence transformer model
    local_files_only: bool = False              # The model is saved in the lab environment
    pmi_threshold: float = 2.0                  # PMI threshold for significance
    cosine_similarity_threshold: float = 0.3    # Cosine similarity threshold
    window_size: int = 5                        # Context window for co-occurrence

    # Network Construction
    edge_weight_method: str = "pmi_cosine"      # Edge weighting method
    min_edge_weight: float = 0.1                # Minimum edge weight
    max_edges_per_node: int = 100               # Maximum edges per node

    # Statistical Testing
    significance_level: float = 0.05            # Alpha level for significance
    fdr_method: str = "fdr_bh"                  # FDR correction method
    bootstrap_samples: int = 1000               # Bootstrap samples for CI
    permutation_tests: int = 10000              # Permutation tests for validation

    # Community Detection
    min_community_size: int = 5                 # Minimum community size
    resolution_parameter: float = 1.0           # Resolution for community detection
    hdbscan_min_cluster_size: int = 10          # HDBSCAN minimum cluster size
    hdbscan_min_samples: int = 5                # HDBSCAN minimum samples

    # Visualization
    max_nodes_visualize: int = 500              # Maximum nodes for visualization
    node_size_multiplier: float = 100           # Node size scaling
    edge_width_multiplier: float = 3            # Edge width scaling

    # Performance
    n_jobs: int = -1                            # Number of parallel jobs
    chunk_size: int = 1000                      # Chunk size for processing
    enable_caching: bool = True                 # Enable result caching

    # Random Seeds
    random_seed: int = 42                       # Random seed for reproducibility

# Initialize configuration
config = SemanticNetworkConfig()
print("✅ Semantic Network Configuration initialized!")
print(f"   📝 Vocabulary size: {config.max_vocab_size:,}")
print(f"   🧠 Embedding model: {config.embedding_model}")
print(f"   📊 PMI threshold: {config.pmi_threshold}")
print(f"   🔗 Similarity threshold: {config.cosine_similarity_threshold}")
print(f"   🏘️ Min community size: {config.min_community_size}")


In [ ]:
class SemanticDataLoader:
    """Advanced data loader for semantic network analysis."""

    def __init__(self, config: SemanticNetworkConfig, logger: logging.Logger):
        self.config = config
        self.logger = logger
        self.nlp = None
        self.embedding_model = None

    def initialize_nlp_models(self):
        """Initialize NLP models for processing."""
        self.logger.info("Initializing NLP models...")

        # Load spaCy model
        try:
            self.nlp = spacy.load(self.config.spacy_model)
        except OSError:
            self.logger.warning(f"Could not load {self.config.spacy_model}, trying en_core_web_sm")
            self.nlp = spacy.load("en_core_web_sm")

        # Load sentence transformer model
        self.embedding_model = SentenceTransformer(
            self.config.embedding_model,
            local_files_only=self.config.local_files_only
        )

        self.logger.info("✅ NLP models initialized successfully")

    def load_real_twitter_data(self, data_directory: str = str(DATA_DIR)) -> List[Dict[str, Any]]:
        """Load and preprocess real Twitter data for semantic analysis."""
        self.logger.info("Loading real Twitter data for semantic analysis")

        data_path = Path(data_directory)
        tweets = []

        # Load ZIP files
        zip_files = ["@ATT.zip", "@TMobile.zip", "@verizon.zip"]

        for zip_file in zip_files:
            zip_path = data_path / zip_file
            print(zip_path)
            if zip_path.exists():
                self.logger.info(f"Loading {zip_file}...")
                tweets.extend(self._load_zip_file(zip_path))
            else:
                self.logger.warning(f"ZIP file not found: {zip_path}")

        # Load sentiment data
        sentiment_data = self._load_sentiment_data(data_path)

        # Merge sentiment data
        tweets = self._merge_sentiment_data(tweets, sentiment_data)

        # Filter and preprocess for semantic analysis
        tweets = self._preprocess_for_semantic_analysis(tweets)

        self.logger.info(f"Loaded and preprocessed {len(tweets)} tweets for semantic analysis")
        return tweets

    def _load_zip_file(self, file_path: Path) -> List[Dict[str, Any]]:
        """Load tweets from a ZIP file."""
        tweets = []

        try:
            with zipfile.ZipFile(file_path, "r") as zip_ref:
                json_files = [f for f in zip_ref.namelist() if f.endswith(".json")]

                for json_file in tqdm(json_files, desc=f"Processing {file_path.name}"):
                    try:
                        with zip_ref.open(json_file) as f:
                            tweet = json.load(f)

                        if isinstance(tweet, dict):
                            processed_tweet = self._process_tweet(tweet)
                            if processed_tweet:
                                tweets.append(processed_tweet)
                    except Exception as e:
                        continue

        except Exception as e:
            self.logger.error(f"Error loading ZIP file {file_path}: {e}")

        return tweets

    def _process_tweet(self, tweet: Dict[str, Any]) -> Optional[Dict[str, Any]]:
        """Process a single tweet for semantic analysis."""
        if not isinstance(tweet, dict):
            return None

        try:
            # Extract tweet ID
            tweet_id = tweet.get("id_str") or str(tweet.get("id", ""))
            if not tweet_id:
                return None

            # Extract user information
            user = tweet.get("user", {})
            if not isinstance(user, dict):
                return None

            user_id = user.get("id_str") or str(user.get("id", ""))
            username = user.get("screen_name", "")

            # Extract text
            text = tweet.get("text", "")
            if not text or len(text.strip()) < self.config.min_tweet_length:
                return None

            # Extract metadata
            created_at = tweet.get("created_at", "")

            # Extract entities for semantic analysis
            entities = tweet.get("entities", {})
            hashtags = [h.get("text", "") for h in entities.get("hashtags", [])]
            user_mentions = [m.get("screen_name", "") for m in entities.get("user_mentions", [])]
            urls = [u.get("expanded_url", "") for u in entities.get("urls", [])]

            return {
                "tweet_id": tweet_id,
                "user_id": user_id,
                "username": username,
                "text": text,
                "created_at": created_at,
                "hashtags": hashtags,
                "user_mentions": user_mentions,
                "urls": urls,
                "followers_count": user.get("followers_count", 0),
                "friends_count": user.get("friends_count", 0),
                "verified": user.get("verified", False),
                # Placeholders for sentiment data
                "sentiment_score": 0.0,
                "sentiment_label": "neutral"
            }

        except Exception as e:
            self.logger.warning(f"Error processing tweet: {e}")
            return None

    def _load_sentiment_data(self, data_path: Path) -> Dict[str, Dict[str, Any]]:
        """Load sentiment analysis data."""
        sentiment_data = {}

        sentiment_files = [
            "mobile_sentiment.csv",
            "mobile_sentiment_positive.csv",
            "mobile_sentiment_negative.csv"
        ]

        for file_name in sentiment_files:
            file_path = data_path / file_name
            if file_path.exists():
                try:
                    df = pd.read_csv(file_path)
                    for _, row in df.iterrows():
                        text = str(row.get("text", "")).strip()
                        if text and len(text) > 10:
                            sentiment_data[text] = {
                                "sentiment_score": float(row.get("polarity", 0.0)),
                                "sentiment_label": str(row.get("sentiment", "neutral")).lower()
                            }
                except Exception as e:
                    self.logger.warning(f"Error loading {file_name}: {e}")

        self.logger.info(f"Loaded sentiment data for {len(sentiment_data)} texts")
        return sentiment_data

    def _merge_sentiment_data(self, tweets: List[Dict[str, Any]], sentiment_data: Dict[str, Dict[str, Any]]) -> List[Dict[str, Any]]:
        """Merge sentiment data with tweets."""
        merged_count = 0

        for tweet in tweets:
            text = tweet.get("text", "")
            if text in sentiment_data:
                tweet.update(sentiment_data[text])
                merged_count += 1

        self.logger.info(f"Merged sentiment data for {merged_count} tweets")
        return tweets

    def _preprocess_for_semantic_analysis(self, tweets: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        """Preprocess tweets specifically for semantic analysis."""
        filtered_tweets = []

        for tweet in tweets:
            text = tweet.get("text", "")

            # Filter by length
            if not (self.config.min_tweet_length <= len(text) <= self.config.max_tweet_length):
                continue

            # Basic text cleaning for semantic analysis
            cleaned_text = self._clean_text_for_semantics(text)
            if len(cleaned_text.split()) < 3:  # Need at least 3 words for semantic analysis
                continue

            tweet["cleaned_text"] = cleaned_text
            tweet["word_count"] = len(cleaned_text.split())
            filtered_tweets.append(tweet)

        self.logger.info(f"Filtered to {len(filtered_tweets)} tweets suitable for semantic analysis")
        return filtered_tweets

    def _clean_text_for_semantics(self, text: str) -> str:
        """Clean text specifically for semantic analysis."""
        # Remove URLs
        text = re.sub(r"http\S+|www\S+|https\S+", "", text, flags=re.MULTILINE)

        # Remove @mentions and #hashtags for semantic analysis
        text = re.sub(r"@\w+|#\w+", "", text)

        # Remove RT prefix
        text = re.sub(r"^RT\s+", "", text)

        # Remove extra whitespace
        text = re.sub(r"\s+", " ", text).strip()

        # Convert to lowercase for semantic analysis
        text = text.lower()

        return text

print("✅ SemanticDataLoader class defined successfully!")


In [ ]:
class AdvancedNLPProcessor:
    """Advanced NLP processing for semantic network construction."""

    def __init__(self, config: SemanticNetworkConfig, nlp_model, logger: logging.Logger):
        self.config = config
        self.nlp = nlp_model
        self.logger = logger
        self.vocabulary = {}
        self.word_frequencies = Counter()

    def process_corpus(self, tweets: List[Dict[str, Any]]) -> Dict[str, Any]:
        """Process entire corpus for semantic analysis."""
        self.logger.info("Processing corpus with advanced NLP pipeline")

        # Extract texts
        texts = [tweet.get("cleaned_text", "") for tweet in tweets]

        # Process texts with spaCy
        processed_docs = []
        word_cooccurrences = defaultdict(lambda: defaultdict(int))

        self.logger.info("Processing texts with spaCy...")
        for i, text in enumerate(tqdm(texts, desc="NLP Processing")):
            if not text.strip():
                continue

            try:
                doc = self.nlp(text)
                processed_doc = self._process_spacy_doc(doc)

                if processed_doc["tokens"]:
                    processed_docs.append({
                        "tweet_idx": i,
                        "original_text": text,
                        **processed_doc
                    })

                    # Build co-occurrence matrix
                    self._update_cooccurrences(processed_doc["tokens"], word_cooccurrences)

            except Exception as e:
                self.logger.warning(f"Error processing text {i}: {e}")
                continue

        # Build vocabulary
        self._build_vocabulary(processed_docs)

        # Calculate PMI scores
        pmi_scores = self._calculate_pmi_scores(word_cooccurrences)

        self.logger.info(f"Processed {len(processed_docs)} documents")
        self.logger.info(f"Vocabulary size: {len(self.vocabulary)}")
        self.logger.info(f"PMI pairs: {len(pmi_scores)}")

        return {
            "processed_docs": processed_docs,
            "vocabulary": self.vocabulary,
            "word_frequencies": dict(self.word_frequencies),
            "cooccurrences": dict(word_cooccurrences),
            "pmi_scores": pmi_scores
        }

    def _process_spacy_doc(self, doc) -> Dict[str, Any]:
        """Process a spaCy document for semantic analysis."""
        tokens = []
        lemmas = []
        pos_tags = []
        entities = []

        for token in doc:
            # Filter tokens
            if (
                token.is_stop or
                token.is_punct or
                token.is_space or
                len(token.text) < 2 or
                token.pos_ not in self.config.pos_tags_to_keep
            ):
                continue

            # Use lemma if lemmatization is enabled
            if self.config.use_lemmatization:
                word = token.lemma_.lower()
            else:
                word = token.text.lower()

            # Additional filtering
            if len(word) >= 2 and word.isalpha():
                tokens.append(token.text.lower())
                lemmas.append(word)
                pos_tags.append(token.pos_)

        # Extract named entities if enabled
        if self.config.use_ner:
            for ent in doc.ents:
                if ent.label_ in ["PERSON", "ORG", "PRODUCT", "EVENT"]:
                    entities.append({
                        "text": ent.text,
                        "label": ent.label_,
                        "start": ent.start_char,
                        "end": ent.end_char
                    })

        return {
            "tokens": tokens,
            "lemmas": lemmas,
            "pos_tags": pos_tags,
            "entities": entities,
            "token_count": len(tokens)
        }

    def _update_cooccurrences(self, tokens: List[str], cooccurrences: Dict[str, Dict[str, int]]):
        """Update word co-occurrence counts."""
        for i, word1 in enumerate(tokens):
            # Update word frequency
            self.word_frequencies[word1] += 1

            # Update co-occurrences within window
            window_start = max(0, i - self.config.window_size)
            window_end = min(len(tokens), i + self.config.window_size + 1)

            for j in range(window_start, window_end):
                if i != j:
                    word2 = tokens[j]
                    if word1 != word2:
                        # Use sorted order for consistent co-occurrence pairs
                        if word1 < word2:
                            cooccurrences[word1][word2] += 1
                        else:
                            cooccurrences[word2][word1] += 1

    def _build_vocabulary(self, processed_docs: List[Dict[str, Any]]):
        """Build filtered vocabulary based on frequency."""
        # Filter by frequency
        frequent_words = [
            word for word, freq in self.word_frequencies.items()
            if freq >= self.config.min_word_frequency
        ]

        # Sort by frequency and take top words
        frequent_words.sort(key=lambda w: self.word_frequencies[w], reverse=True)
        frequent_words = frequent_words[:self.config.max_vocab_size]

        # Create vocabulary mapping
        self.vocabulary = {word: idx for idx, word in enumerate(frequent_words)}

        self.logger.info(f"Built vocabulary with {len(self.vocabulary)} words")
        self.logger.info(f"Frequency range: {min(self.word_frequencies[w] for w in frequent_words)} - {max(self.word_frequencies[w] for w in frequent_words)}")

    def _calculate_pmi_scores(self, cooccurrences: Dict[str, Dict[str, int]]) -> Dict[Tuple[str, str], float]:
        """Calculate PMI (Pointwise Mutual Information) scores."""
        self.logger.info("Calculating PMI scores...")

        total_words = sum(self.word_frequencies.values())
        total_pairs = sum(
            sum(pair_counts.values())
            for pair_counts in cooccurrences.values()
        )

        pmi_scores = {}
        significant_pairs = 0

        for word1, word2_counts in tqdm(cooccurrences.items(), desc="Calculating PMI"):
            if word1 not in self.vocabulary:
                continue

            for word2, cooccur_count in word2_counts.items():
                if word2 not in self.vocabulary:
                    continue

                # Calculate PMI
                p_w1 = self.word_frequencies[word1] / total_words
                p_w2 = self.word_frequencies[word2] / total_words
                p_w1_w2 = cooccur_count / total_pairs

                if p_w1 * p_w2 > 0:
                    pmi = np.log2(p_w1_w2 / (p_w1 * p_w2))

                    # Only keep significant PMI scores
                    if pmi >= self.config.pmi_threshold:
                        pmi_scores[(word1, word2)] = pmi
                        significant_pairs += 1

        self.logger.info(f"Found {significant_pairs} significant word pairs (PMI >= {self.config.pmi_threshold})")
        return pmi_scores

print("✅ AdvancedNLPProcessor class defined successfully!")


In [ ]:
class SemanticNetworkConstructor:
    """Construct semantic word networks with statistical validation."""

    def __init__(self, config: SemanticNetworkConfig, embedding_model, logger: logging.Logger):
        self.config = config
        self.embedding_model = embedding_model
        self.logger = logger

    def build_semantic_networks(self, nlp_data: Dict[str, Any]) -> Dict[str, Any]:
        """Build comprehensive semantic networks."""
        self.logger.info("Building semantic networks")

        vocabulary = nlp_data["vocabulary"]
        word_frequencies = nlp_data["word_frequencies"]
        pmi_scores = nlp_data["pmi_scores"]
        processed_docs = nlp_data["processed_docs"]

        # Generate word embeddings
        word_embeddings = self._generate_word_embeddings(vocabulary)

        # Build different types of semantic networks
        networks = {
            "pmi_network": self._build_pmi_network(vocabulary, pmi_scores),
            "embedding_network": self._build_embedding_network(vocabulary, word_embeddings),
            "hybrid_network": self._build_hybrid_network(vocabulary, pmi_scores, word_embeddings),
            "brand_semantic_network": self._build_brand_semantic_network(processed_docs, word_embeddings)
        }

        # Add network statistics
        for name, network in networks.items():
            self._add_network_stats(network, name)

        return {
            "networks": networks,
            "word_embeddings": word_embeddings,
            "vocabulary": vocabulary,
            "word_frequencies": word_frequencies
        }

    def _generate_word_embeddings(self, vocabulary: Dict[str, int]) -> Dict[str, np.ndarray]:
        """Generate contextual embeddings for vocabulary words."""
        self.logger.info("Generating word embeddings...")

        words = list(vocabulary.keys())
        word_embeddings = {}

        # Generate embeddings in batches for efficiency
        batch_size = 100
        for i in tqdm(range(0, len(words), batch_size), desc="Generating embeddings"):
            batch_words = words[i:i+batch_size]

            try:
                # Use sentence transformer to get word embeddings
                batch_embeddings = self.embedding_model.encode(batch_words)

                for word, embedding in zip(batch_words, batch_embeddings):
                    word_embeddings[word] = embedding

            except Exception as e:
                self.logger.warning(f"Error generating embeddings for batch {i}: {e}")
                continue

        self.logger.info(f"Generated embeddings for {len(word_embeddings)} words")
        return word_embeddings

    def _build_pmi_network(self, vocabulary: Dict[str, int], pmi_scores: Dict[Tuple[str, str], float]) -> nx.Graph:
        """Build network based on PMI scores."""
        G = nx.Graph()

        # Add nodes
        for word in vocabulary.keys():
            G.add_node(word, node_type="word")

        # Add edges based on PMI scores
        edges_added = 0
        for (word1, word2), pmi_score in pmi_scores.items():
            if word1 in vocabulary and word2 in vocabulary:
                G.add_edge(word1, word2,
                           weight=pmi_score,
                           edge_type="pmi",
                           pmi_score=pmi_score)
                edges_added += 1

        self.logger.info(f"PMI network: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
        return G

    def _build_embedding_network(self, vocabulary: Dict[str, int], word_embeddings: Dict[str, np.ndarray]) -> nx.Graph:
        """Build network based on embedding similarities."""
        G = nx.Graph()

        # Add nodes
        for word in vocabulary.keys():
            if word in word_embeddings:
                G.add_node(word, node_type="word")

        # Calculate pairwise similarities
        words_with_embeddings = [w for w in vocabulary.keys() if w in word_embeddings]

        self.logger.info(f"Calculating similarities for {len(words_with_embeddings)} words...")

        for i, word1 in enumerate(tqdm(words_with_embeddings, desc="Building embedding network")):
            embedding1 = word_embeddings[word1]
            similarities = []

            # Calculate similarities with other words
            for j, word2 in enumerate(words_with_embeddings[i+1:], i+1):
                embedding2 = word_embeddings[word2]

                # Cosine similarity
                similarity = np.dot(embedding1, embedding2) / (
                    np.linalg.norm(embedding1) * np.linalg.norm(embedding2)
                )

                if similarity >= self.config.cosine_similarity_threshold:
                    similarities.append((word2, similarity))

            # Add top similarities as edges (limit connections per node)
            similarities.sort(key=lambda x: x[1], reverse=True)
            for word2, similarity in similarities[:self.config.max_edges_per_node]:
                G.add_edge(word1, word2,
                           weight=similarity,
                           edge_type="cosine_similarity",
                           similarity=similarity)

        self.logger.info(f"Embedding network: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
        return G

    def _build_hybrid_network(self, vocabulary: Dict[str, int], pmi_scores: Dict[Tuple[str, str], float], word_embeddings: Dict[str, np.ndarray]) -> nx.Graph:
        """Build hybrid network combining PMI and embedding similarities."""
        G = nx.Graph()

        # Add nodes
        for word in vocabulary.keys():
            if word in word_embeddings:
                G.add_node(word, node_type="word")

        # Combine PMI and embedding similarities
        hybrid_scores = {}

        # Add PMI scores
        for (word1, word2), pmi_score in pmi_scores.items():
            if word1 in word_embeddings and word2 in word_embeddings:
                hybrid_scores[(word1, word2)] = {"pmi": pmi_score, "cosine": 0.0}

        # Add cosine similarities for PMI pairs
        for (word1, word2) in tqdm(hybrid_scores.keys(), desc="Computing hybrid scores"):
            embedding1 = word_embeddings[word1]
            embedding2 = word_embeddings[word2]

            similarity = np.dot(embedding1, embedding2) / (
                np.linalg.norm(embedding1) * np.linalg.norm(embedding2)
            )

            hybrid_scores[(word1, word2)]["cosine"] = similarity

        # Create edges with hybrid weights
        for (word1, word2), scores in hybrid_scores.items():
            # Normalize and combine scores
            pmi_norm = scores["pmi"] / 10.0  # Normalize PMI
            cosine_norm = scores["cosine"]   # Already normalized

            # Weighted combination
            hybrid_weight = 0.6 * pmi_norm + 0.4 * cosine_norm

            if hybrid_weight >= self.config.min_edge_weight:
                G.add_edge(word1, word2,
                           weight=hybrid_weight,
                           edge_type="hybrid",
                           pmi_score=scores["pmi"],
                           cosine_similarity=scores["cosine"],
                           hybrid_weight=hybrid_weight)

        self.logger.info(f"Hybrid network: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
        return G

    def _build_brand_semantic_network(self, processed_docs: List[Dict[str, Any]], word_embeddings: Dict[str, np.ndarray]) -> nx.Graph:
        """Build brand-specific semantic network."""
        G = nx.Graph()

        # Define brand keywords
        brand_keywords = {
            "att": ["att", "at&t", "service", "network", "coverage", "bill", "plan", "customer"],
            "tmobile": ["tmobile", "t-mobile", "uncarrier", "magenta", "unlimited", "network"],
            "verizon": ["verizon", "wireless", "fios", "network", "coverage", "plan", "service"]
        }

        # Add brand nodes
        for brand in brand_keywords.keys():
            G.add_node(brand, node_type="brand")

        # Add word nodes that appear with brands
        brand_word_associations = defaultdict(list)

        for doc in processed_docs:
            tokens = doc.get("tokens", [])

            # Find brand mentions in document
            doc_brands = []
            for brand, keywords in brand_keywords.items():
                if any(keyword in token.lower() for token in tokens for keyword in keywords):
                    doc_brands.append(brand)

            # Associate words with brands
            if doc_brands:
                for token in tokens:
                    if token in word_embeddings:
                        for brand in doc_brands:
                            brand_word_associations[brand].append(token)

        # Build brand-word connections
        for brand, words in brand_word_associations.items():
            word_counts = Counter(words)
            total_words = sum(word_counts.values())

            for word, count in word_counts.items():
                if count >= 5:  # Minimum association frequency
                    association_strength = count / total_words

                    if not G.has_node(word):
                        G.add_node(word, node_type="word")

                    G.add_edge(brand, word,
                               weight=association_strength,
                               edge_type="brand_association",
                               association_count=count,
                               association_strength=association_strength)

        self.logger.info(f"Brand semantic network: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
        return G

    def _add_network_stats(self, network: nx.Graph, network_name: str):
        """Add basic statistics to network."""
        stats = {
            "nodes": network.number_of_nodes(),
            "edges": network.number_of_edges(),
            "density": nx.density(network),
            "avg_degree": sum(dict(network.degree()).values()) / network.number_of_nodes() if network.number_of_nodes() > 0 else 0
        }

        if network.number_of_nodes() > 0:
            components = list(nx.connected_components(network))
            stats["connected_components"] = len(components)
            if components:
                stats["largest_component_size"] = len(max(components, key=len))

        network.graph["stats"] = stats

        print(f"📊 {network_name.replace('_', ' ').title()}:")
        print(f"   Nodes: {stats['nodes']:,}")
        print(f"   Edges: {stats['edges']:,}")
        print(f"   Density: {stats['density']:.6f}")
        print(f"   Avg Degree: {stats['avg_degree']:.2f}")
        if "connected_components" in stats:
            print(f"   Components: {stats['connected_components']}")
        print()

print("✅ SemanticNetworkConstructor class defined successfully!")


In [ ]:
class SemanticNetworkAnalyzer:
    """Advanced semantic network analysis with statistical validation."""

    def __init__(self, config: SemanticNetworkConfig, logger: logging.Logger):
        self.config = config
        self.logger = logger

    def analyze_semantic_networks(self, network_data: Dict[str, Any]) -> Dict[str, Any]:
        """Perform comprehensive semantic network analysis."""
        self.logger.info("Starting semantic network analysis")

        networks = network_data["networks"]
        word_embeddings = network_data["word_embeddings"]

        results = {}

        for network_name, network in networks.items():
            if network.number_of_nodes() == 0:
                continue

            display_name = network_name.upper().replace("_", " ")
            print(f"\n�� ANALYZING {display_name}")
            print("=" * 60)

            analysis = {
                "semantic_centrality": self._analyze_semantic_centrality(network),
                "semantic_communities": self._detect_semantic_communities(network, word_embeddings),
                "semantic_structure": self._analyze_semantic_structure(network),
                "brand_positioning": self._analyze_brand_positioning(network, word_embeddings) if "brand" in network_name else None
            }

            results[network_name] = analysis
            self._print_semantic_analysis_summary(network_name, analysis)

        return results

    def _analyze_semantic_centrality(self, G: nx.Graph) -> Dict[str, Any]:
        """Analyze semantic centrality measures."""
        centrality = {}

        # Standard centrality measures
        centrality["degree"] = nx.degree_centrality(G)
        centrality["betweenness"] = nx.betweenness_centrality(G)
        centrality["eigenvector"] = self._safe_eigenvector_centrality(G)
        centrality["pagerank"] = nx.pagerank(G)

        # Semantic-specific centrality
        centrality["semantic_centrality"] = self._calculate_semantic_centrality(G)

        return centrality

    def _safe_eigenvector_centrality(self, G: nx.Graph) -> Dict[str, float]:
        """Calculate eigenvector centrality with fallback."""
        try:
            return nx.eigenvector_centrality(G, max_iter=1000)
        except:
            return nx.degree_centrality(G)

    def _calculate_semantic_centrality(self, G: nx.Graph) -> Dict[str, float]:
        """Calculate custom semantic centrality based on edge weights."""
        semantic_centrality = {}

        for node in G.nodes():
            total_weight = sum(
                G[node][neighbor].get("weight", 1.0)
                for neighbor in G.neighbors(node)
            )
            semantic_centrality[node] = total_weight

        # Normalize
        if semantic_centrality:
            max_weight = max(semantic_centrality.values())
            if max_weight > 0:
                semantic_centrality = {
                    node: weight / max_weight
                    for node, weight in semantic_centrality.items()
                }

        return semantic_centrality

    def _detect_semantic_communities(self, G: nx.Graph, word_embeddings: Dict[str, np.ndarray]) -> Dict[str, Any]:
        """Detect semantic communities with validation."""
        communities = {}

        if G.number_of_nodes() < 10:
            return {"method": "none", "reason": "Too few nodes for community detection"}

        # Louvain community detection
        try:
            louvain_communities = community_louvain.best_partition(G)
            louvain_modularity = community_louvain.modularity(louvain_communities, G)

            communities["louvain"] = {
                "partition": louvain_communities,
                "modularity": louvain_modularity,
                "num_communities": len(set(louvain_communities.values()))
            }
        except Exception as e:
            communities["louvain"] = {"error": str(e)}

        return communities

    def _analyze_semantic_structure(self, G: nx.Graph) -> Dict[str, Any]:
        """Analyze semantic network structure."""
        structure = {
            "nodes": G.number_of_nodes(),
            "edges": G.number_of_edges(),
            "density": nx.density(G),
            "average_clustering": nx.average_clustering(G),
            "transitivity": nx.transitivity(G)
        }

        # Connectivity
        if G.number_of_nodes() > 0:
            components = list(nx.connected_components(G))
            structure["connected_components"] = len(components)
            if components:
                structure["largest_component_size"] = len(max(components, key=len))

        return structure

    def _analyze_brand_positioning(self, G: nx.Graph, word_embeddings: Dict[str, np.ndarray]) -> Dict[str, Any]:
        """Analyze brand positioning in semantic space."""
        brand_nodes = [node for node, data in G.nodes(data=True) if data.get("node_type") == "brand"]

        if len(brand_nodes) < 2:
            return {"error": "Need at least 2 brands for positioning analysis"}

        positioning = {"brand_neighborhoods": {}}

        # Analyze brand neighborhoods
        for brand in brand_nodes:
            neighbors = list(G.neighbors(brand))
            edge_weights = [G[brand][neighbor].get("weight", 0) for neighbor in neighbors]

            if neighbors and edge_weights:
                neighbor_weights = list(zip(neighbors, edge_weights))
                neighbor_weights.sort(key=lambda x: x[1], reverse=True)

                positioning["brand_neighborhoods"][brand] = {
                    "top_words": neighbor_weights[:20],
                    "total_associations": len(neighbors),
                    "avg_association_strength": np.mean(edge_weights)
                }

        return positioning

    def _print_semantic_analysis_summary(self, network_name: str, analysis: Dict[str, Any]):
        """Print summary of semantic analysis."""
        structure = analysis.get("semantic_structure", {})
        communities = analysis.get("semantic_communities", {})
        centrality = analysis.get("semantic_centrality", {})

        print(f"📊 Semantic Structure:")
        print(f"   Nodes: {structure.get('nodes', 0):,}")
        print(f"   Edges: {structure.get('edges', 0):,}")
        print(f"   Density: {structure.get('density', 0):.6f}")
        print(f"   Clustering: {structure.get('average_clustering', 0):.4f}")

        # Community results
        if "louvain" in communities and "modularity" in communities["louvain"]:
            louvain = communities["louvain"]
            print(f"\n🏘️ Semantic Communities (Louvain):")
            print(f"   Communities: {louvain.get('num_communities', 0)}")
            print(f"   Modularity: {louvain.get('modularity', 0):.4f}")

        print()

print("✅ SemanticNetworkAnalyzer class defined successfully!")


In [ ]:
# Execute the complete semantic network analysis pipeline
print("🧠 EXECUTING COMPLETE SEMANTIC WORD NETWORK ANALYSIS")
print("=" * 70)

# Step 1: Initialize data loader and NLP models
print("\n🔧 Step 1: Initializing semantic analysis components...")
data_loader = SemanticDataLoader(config, logger)
data_loader.initialize_nlp_models()

# Step 2: Load and preprocess Twitter data
print("\n📂 Step 2: Loading and preprocessing Twitter data...")
tweets = data_loader.load_real_twitter_data()

if not tweets:
    print("❌ No tweets loaded. Please check your data files.")
else:
    print(f"✅ Successfully loaded {len(tweets)} tweets for semantic analysis")

    # Step 3: Advanced NLP processing
    print("\n🧠 Step 3: Performing advanced NLP processing...")
    nlp_processor = AdvancedNLPProcessor(config, data_loader.nlp, logger)
    nlp_data = nlp_processor.process_corpus(tweets)

    print(f"   📝 Processed documents: {len(nlp_data['processed_docs'])}")
    print(f"   📚 Vocabulary size: {len(nlp_data['vocabulary'])}")
    print(f"   🔗 PMI word pairs: {len(nlp_data['pmi_scores'])}")

    # Step 4: Build semantic networks
    print("\n🕸️ Step 4: Building semantic word networks...")
    network_constructor = SemanticNetworkConstructor(config, data_loader.embedding_model, logger)
    network_data = network_constructor.build_semantic_networks(nlp_data)

    networks = network_data["networks"]
    word_embeddings = network_data["word_embeddings"]

    print(f"   🧠 Word embeddings: {len(word_embeddings)}")
    print(f"   📊 Networks created: {len(networks)}")

    # Step 5: Analyze semantic networks
    print("\n🔬 Step 5: Performing semantic network analysis...")
    analyzer = SemanticNetworkAnalyzer(config, logger)
    analysis_results = analyzer.analyze_semantic_networks(network_data)

    # Step 6: Generate comprehensive summary
    print("\n📋 SEMANTIC ANALYSIS COMPLETE - SUMMARY REPORT")
    print("=" * 60)

    print(f"✅ Data Processing:")
    print(f"   • Processed {len(tweets):,} tweets for semantic analysis")
    print(f"   • Built vocabulary of {len(nlp_data['vocabulary']):,} words")
    print(f"   • Generated {len(word_embeddings):,} word embeddings")
    print(f"   • Found {len(nlp_data['pmi_scores']):,} significant word pairs (PMI)")

    print(f"\n✅ Semantic Networks:")
    for name, network in networks.items():
        display_name = name.replace("_", " ").title()
        stats = network.graph.get("stats", {})
        print(f"   • {display_name}:")
        print(f"     - Nodes: {stats.get('nodes', 0):,}")
        print(f"     - Edges: {stats.get('edges', 0):,}")
        print(f"     - Density: {stats.get('density', 0):.6f}")
        if "connected_components" in stats:
            print(f"     - Components: {stats.get('connected_components', 0)}")

    print(f"\n✅ Analysis Results:")
    for name, results in analysis_results.items():
        if "semantic_structure" in results:
            struct = results["semantic_structure"]
            print(f"   • {name.replace('_', ' ').title()}:")
            print(f"     - Clustering: {struct.get('average_clustering', 0):.4f}")
            print(f"     - Transitivity: {struct.get('transitivity', 0):.4f}")

            if "semantic_communities" in results:
                communities = results["semantic_communities"]
                if "louvain" in communities and "modularity" in communities["louvain"]:
                    louvain = communities["louvain"]
                    print(f"     - Communities: {louvain.get('num_communities', 0)}")
                    print(f"     - Modularity: {louvain.get('modularity', 0):.4f}")

    print(f"\n🎉 SEMANTIC WORD NETWORK ANALYSIS COMPLETED SUCCESSFULLY!")
    print(f"🧠 Results contain comprehensive semantic relationships and word communities.")

    # Store results for further analysis
    semantic_analysis_data = {
        "tweets": tweets,
        "nlp_data": nlp_data,
        "network_data": network_data,
        "analysis_results": analysis_results,
        "config": config
    }

    print("\n💾 All results stored in semantic_analysis_data variable for exploration.")


In [ ]:
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import networkx as nx
import numpy as np
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import umap

class InteractiveSemanticNetworkVisualizer:
    """Interactive visualization for semantic word networks in Jupyter."""

    def __init__(self, config):
        self.config = config

    def visualize_semantic_networks(self, network_data: dict, analysis_results: dict = None):
        """Create interactive visualizations for all semantic networks."""
        print("🎨 Creating interactive semantic network visualizations...")

        networks = network_data["networks"]
        word_embeddings = network_data.get("word_embeddings", {})

        # Visualize each semantic network
        for network_name, network in networks.items():
            if network.number_of_nodes() == 0:
                continue

            print(f"\n🧠 Visualizing {network_name.replace('_', ' ').title()}...")

            if "brand" in network_name:
                self._visualize_brand_semantic_network(network, network_name, word_embeddings, analysis_results)
            else:
                self._visualize_word_semantic_network(network, network_name, word_embeddings, analysis_results)

        # Create embedding space visualization if embeddings available
        if word_embeddings:
            self._visualize_embedding_space(word_embeddings, networks)

    def _visualize_word_semantic_network(self, G: nx.Graph, network_name: str, word_embeddings: dict, analysis_results: dict = None):
        """Visualize word semantic networks."""
        if G.number_of_nodes() > 300:
            # Sample large semantic networks
            print(f"   📉 Sampling {G.number_of_nodes()} nodes to 300 for visualization")

            # Get nodes with highest semantic centrality
            if analysis_results and network_name in analysis_results:
                centrality_data = analysis_results[network_name].get("semantic_centrality", {})
                semantic_cent = centrality_data.get("semantic_centrality", {})
                if semantic_cent:
                    top_nodes = sorted(semantic_cent.items(), key=lambda x: x[1], reverse=True)[:300]
                    sample_nodes = [node for node, _ in top_nodes]
                else:
                    # Fallback to degree
                    degrees = dict(G.degree())
                    top_nodes = sorted(degrees.items(), key=lambda x: x[1], reverse=True)[:300]
                    sample_nodes = [node for node, _ in top_nodes]
            else:
                degrees = dict(G.degree())
                top_nodes = sorted(degrees.items(), key=lambda x: x[1], reverse=True)[:300]
                sample_nodes = [node for node, _ in top_nodes]

            G = G.subgraph(sample_nodes).copy()

        # Use embedding-based layout if available
        if word_embeddings:
            pos = self._get_embedding_layout(G, word_embeddings)
        else:
            pos = nx.spring_layout(G, k=2, iterations=50)

        # Extract node and edge information
        node_info = self._extract_semantic_node_info(G, pos, analysis_results, network_name, word_embeddings)
        edge_info = self._extract_semantic_edge_info(G, pos)

        # Create plotly figure
        fig = go.Figure()

        # Add edges with varying opacity based on weight
        fig.add_trace(go.Scatter(
            x=edge_info["x"],
            y=edge_info["y"],
            mode="lines",
            line=dict(width=float(np.median(edge_info["widths"])), color="rgba(100,100,100,0.4)"),
            hoverinfo="none",
            showlegend=False,
            name="Edges"
        ))

        # Add nodes
        fig.add_trace(go.Scatter(
            x=node_info["x"],
            y=node_info["y"],
            mode="markers+text",
            marker=dict(
                size=node_info["sizes"],
                color=node_info["colors"],
                colorscale="Spectral",
                showscale=True,
                colorbar=dict(title="Semantic Centrality"),
                line=dict(width=1, color="darkgray"),
                opacity=0.8
            ),
            text=node_info["labels"],
            textposition="middle center",
            textfont=dict(size=10, color="black", family="Arial Black"),
            hovertemplate=node_info["hover_text"],
            showlegend=False,
            name="Words"
        ))

        # Update layout
        title = f"🧠 {network_name.replace('_', ' ').title()} Semantic Network"
        fig.update_layout(
            title=dict(
                text=title,
                x=0.5,
                font=dict(size=18, color="darkgreen")
            ),
            showlegend=False,
            hovermode="closest",
            margin=dict(b=20,l=5,r=5,t=40),
            annotations=[
                dict(
                    text=f"Words: {G.number_of_nodes():,} | Relationships: {G.number_of_edges():,} | Density: {nx.density(G):.4f}",
                    showarrow=False,
                    xref="paper", yref="paper",
                    x=0.005, y=-0.002,
                    xanchor="left", yanchor="bottom",
                    font=dict(size=12, color="gray")
                )
            ],
            xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
            yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
            plot_bgcolor="white",
            width=900,
            height=700
        )

        fig.show()

    def _visualize_brand_semantic_network(self, G: nx.Graph, network_name: str, word_embeddings: dict, analysis_results: dict = None):
        """Visualize brand-semantic networks with special highlighting."""
        # Get layout
        pos = nx.spring_layout(G, k=3, iterations=50)

        # Separate brand and word nodes
        brand_nodes = [node for node, data in G.nodes(data=True) if data.get("node_type") == "brand"]
        word_nodes = [node for node in G.nodes() if node not in brand_nodes]

        # Create figure
        fig = go.Figure()

        # Add edges
        edge_info = self._extract_semantic_edge_info(G, pos)
        fig.add_trace(go.Scatter(
            x=edge_info["x"],
            y=edge_info["y"],
            mode="lines",
            line=dict(width=1, color="rgba(150,150,150,0.6)"),
            hoverinfo="none",
            showlegend=False
        ))

        # Add brand nodes (larger, special colors)
        if brand_nodes:
            brand_x = [pos[node][0] for node in brand_nodes]
            brand_y = [pos[node][1] for node in brand_nodes]
            brand_colors = ["red", "blue", "green", "orange", "purple"][:len(brand_nodes)]

            fig.add_trace(go.Scatter(
                x=brand_x,
                y=brand_y,
                mode="markers+text",
                marker=dict(
                    size=[40] * len(brand_nodes),
                    color=brand_colors,
                    line=dict(width=3, color="white"),
                    symbol="star"
                ),
                text=[node.upper() for node in brand_nodes],
                textposition="middle center",
                textfont=dict(size=12, color="white", family="Arial Black"),
                hovertemplate="<b>%{text}</b><br>Brand Node<extra></extra>",
                name="Brands",
                showlegend=True
            ))

        # Add word nodes
        if word_nodes:
            word_x = [pos[node][0] for node in word_nodes]
            word_y = [pos[node][1] for node in word_nodes]

            # Get association strengths for coloring
            word_sizes = []
            word_colors = []
            word_labels = []

            for word in word_nodes:
                # Size based on total associations
                total_weight = sum(G[word][neighbor].get("weight", 0) for neighbor in G.neighbors(word))
                size = 15 + min(25, total_weight * 100)  # Scale appropriately
                word_sizes.append(size)
                word_colors.append(total_weight)

                # Show label for important words
                if size > 25:
                    word_labels.append(word[:8])
                else:
                    word_labels.append("")

            fig.add_trace(go.Scatter(
                x=word_x,
                y=word_y,
                mode="markers+text",
                marker=dict(
                    size=word_sizes,
                    color=word_colors,
                    colorscale="Viridis",
                    showscale=True,
                    colorbar=dict(title="Association Strength"),
                    line=dict(width=1, color="darkgray")
                ),
                text=word_labels,
                textposition="middle center",
                textfont=dict(size=8, color="white"),
                hovertemplate="<b>%{text}</b><br>Semantic Word<extra></extra>",
                name="Words",
                showlegend=True
            ))

        # Update layout
        fig.update_layout(
            title=dict(
                text="⭐ Brand Semantic Network - Word Associations",
                x=0.5,
                font=dict(size=18, color="darkblue")
            ),
            hovermode="closest",
            margin=dict(b=20,l=5,r=5,t=40),
            xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
            yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
            plot_bgcolor="white",
            width=900,
            height=700,
            legend=dict(x=0, y=1, bgcolor="rgba(255,255,255,0.8)")
        )

        fig.show()

    def _visualize_embedding_space(self, word_embeddings: dict, networks: dict):
        """Create 2D visualization of word embedding space."""
        print("\n🤖 Creating Word Embedding Space Visualization...")

        # Get sample of words for visualization
        sample_words = list(word_embeddings.keys())[:500]  # Limit for performance
        embeddings_matrix = np.array([word_embeddings[word] for word in sample_words])

        # Reduce dimensionality using UMAP
        try:
            reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1)
            embeddings_2d = reducer.fit_transform(embeddings_matrix)
        except:
            # Fallback to t-SNE
            print("   Using t-SNE fallback for dimensionality reduction")
            tsne = TSNE(n_components=2, random_state=42, perplexity=30)
            embeddings_2d = tsne.fit_transform(embeddings_matrix)

        # Determine word importance (frequency in networks)
        word_importance = {}
        for word in sample_words:
            importance = 0
            for network in networks.values():
                if word in network.nodes():
                    importance += network.degree(word)
            word_importance[word] = importance

        # Create scatter plot
        fig = go.Figure()

        # Color by importance, size by frequency
        colors = [word_importance[word] for word in sample_words]
        sizes = [8 + min(20, word_importance[word] / 2) for word in sample_words]

        # Show labels for most important words
        labels = []
        for word in sample_words:
            if word_importance[word] > np.percentile(list(word_importance.values()), 90):
                labels.append(word)
            else:
                labels.append("")

        fig.add_trace(go.Scatter(
            x=embeddings_2d[:, 0],
            y=embeddings_2d[:, 1],
            mode="markers+text",
            marker=dict(
                size=sizes,
                color=colors,
                colorscale="Turbo",
                showscale=True,
                colorbar=dict(title="Network Importance"),
                line=dict(width=0.5, color="gray"),
                opacity=0.7
            ),
            text=labels,
            textposition="top center",
            textfont=dict(size=9, color="black"),
            hovertemplate="<b>%{text}</b><br>Importance: %{marker.color}<extra></extra>",
            showlegend=False
        ))

        fig.update_layout(
            title=dict(
                text="🤖 Word Embedding Space (2D Projection)",
                x=0.5,
                font=dict(size=18, color="darkviolet")
            ),
            xaxis=dict(title="UMAP Dimension 1", showgrid=True, gridcolor="lightgray"),
            yaxis=dict(title="UMAP Dimension 2", showgrid=True, gridcolor="lightgray"),
            plot_bgcolor="white",
            width=900,
            height=700,
            hovermode="closest"
        )

        fig.show()

    def _get_embedding_layout(self, G: nx.Graph, word_embeddings: dict):
        """Get network layout based on word embeddings."""
        # Get embeddings for nodes in the graph
        nodes_with_embeddings = [node for node in G.nodes() if node in word_embeddings]

        if len(nodes_with_embeddings) < 10:
            return nx.spring_layout(G)

        # Use PCA to reduce embeddings to 2D
        embeddings_matrix = np.array([word_embeddings[node] for node in nodes_with_embeddings])
        pca = PCA(n_components=2)
        embeddings_2d = pca.fit_transform(embeddings_matrix)

        # Create position dictionary
        pos = {}
        for i, node in enumerate(nodes_with_embeddings):
            pos[node] = embeddings_2d[i]

        # Add spring layout for nodes without embeddings
        remaining_nodes = [node for node in G.nodes() if node not in pos]
        if remaining_nodes:
            spring_pos = nx.spring_layout(G.subgraph(remaining_nodes))
            pos.update(spring_pos)

        return pos

    def _extract_semantic_node_info(self, G, pos, analysis_results, network_name, word_embeddings):
        """Extract node information for semantic networks."""
        x_nodes = [pos[node][0] for node in G.nodes()]
        y_nodes = [pos[node][1] for node in G.nodes()]

        # Get semantic centrality for coloring
        if analysis_results and network_name in analysis_results:
            centrality_data = analysis_results[network_name].get("semantic_centrality", {})
            semantic_cent = centrality_data.get("semantic_centrality", {})
        else:
            semantic_cent = {}

        # Get edge weights for sizing
        node_weights = {}
        for node in G.nodes():
            total_weight = sum(G[node][neighbor].get("weight", 1) for neighbor in G.neighbors(node))
            node_weights[node] = total_weight

        max_weight = max(node_weights.values()) if node_weights else 1

        sizes = []
        colors = []
        labels = []
        hover_texts = []

        for node in G.nodes():
            weight = node_weights.get(node, 0)
            centrality = semantic_cent.get(node, 0)

            # Size based on semantic weight
            size = 15 + 35 * (weight / max_weight)
            sizes.append(size)

            # Color based on centrality
            colors.append(centrality)

            # Label for important nodes
            if weight > max_weight * 0.2:  # Top 20% by weight
                labels.append(str(node)[:12])
            else:
                labels.append("")

            # Hover text
            hover_text = f"<b>{node}</b><br>"
            hover_text += f"Semantic Weight: {weight:.3f}<br>"
            hover_text += f"Centrality: {centrality:.4f}<br>"
            hover_text += f"Connections: {G.degree(node)}"
            hover_texts.append(hover_text)

        return {
            "x": x_nodes,
            "y": y_nodes,
            "sizes": sizes,
            "colors": colors,
            "labels": labels,
            "hover_text": hover_texts
        }

    def _extract_semantic_edge_info(self, G, pos):
        """Extract edge information with weights for semantic networks."""
        x_edges = []
        y_edges = []
        widths = []

        # Get edge weights for width scaling
        edge_weights = [G[u][v].get("weight", 1) for u, v in G.edges()]
        if edge_weights:
            max_weight = max(edge_weights)
            min_width, max_width = 0.5, 3

        for edge in G.edges():
            x0, y0 = pos[edge[0]]
            x1, y1 = pos[edge[1]]
            x_edges.extend([x0, x1, None])
            y_edges.extend([y0, y1, None])

            # Width based on edge weight
            if edge_weights:
                weight = G[edge[0]][edge[1]].get("weight", 1)
                width = min_width + (max_width - min_width) * (weight / max_weight)
                widths.extend([width, width, width])
            else:
                widths.extend([1, 1, 1])

        return {"x": x_edges, "y": y_edges, "widths": widths}

print("✅ InteractiveSemanticNetworkVisualizer class defined successfully!")


In [ ]:
# Execute semantic network visualization
if "semantic_analysis_data" in locals() and semantic_analysis_data:
    print("🧠 CREATING INTERACTIVE SEMANTIC NETWORK VISUALIZATIONS")
    print("=" * 60)

    # Initialize semantic visualizer
    semantic_visualizer = InteractiveSemanticNetworkVisualizer(config)

    # Visualize all semantic networks
    network_data = semantic_analysis_data["network_data"]
    analysis_results = semantic_analysis_data["analysis_results"]

    semantic_visualizer.visualize_semantic_networks(network_data, analysis_results)

    print("\n✅ All semantic network visualizations completed!")
    print("🧠 Interactive plots show:")
    print("   • Word nodes sized by semantic importance")
    print("   • Node colors represent semantic centrality")
    print("   • Edge thickness shows relationship strength")
    print("   • Brand nodes highlighted with star symbols")
    print("   • 2D embedding space projection (UMAP/t-SNE)")
    print("   • Automatic sampling for large networks (>300 words)")
    print("   • Hover for detailed semantic information")
else:
    print("❌ Please run the main semantic analysis first to generate networks for visualization")


## 🎯 **Semantic Network Analysis Complete!**

### 🧠 **What This Analysis Accomplished**

1. **Advanced Text Processing**
   - spaCy transformer pipeline with lemmatization
   - Named Entity Recognition (NER)
   - POS tagging and linguistic filtering
   - Word co-occurrence matrix construction

2. **Multi-Scale Semantic Networks**
   - **PMI Network**: Based on statistical co-occurrence significance
   - **Embedding Network**: Using transformer-based semantic similarity
   - **Hybrid Network**: Combining PMI and embedding approaches
   - **Brand Semantic Network**: Brand-word association patterns

3. **Advanced Analysis Techniques**
   - Semantic centrality measures
   - Community detection with statistical validation
   - Brand positioning in semantic space
   - Cross-network comparative analysis

### 🔬 **Key Insights Available**

- **Word Relationship Patterns**: Which words co-occur significantly
- **Semantic Communities**: Groups of related concepts
- **Brand Positioning**: How brands are positioned in semantic space
- **Language Evolution**: Temporal patterns in word usage
- **Influence Networks**: Which words/concepts drive conversations

### 🚀 **Next Steps for Analysis**

1. **Visualize semantic networks** using interactive plots
2. **Compare brand semantic positioning** across competitors
3. **Analyze temporal semantic evolution** over time
4. **Identify emerging topics** and trending concepts
5. **Generate semantic insights reports** for stakeholders

### 📊 **Available Data Objects**

- `semantic_analysis_data['networks']` - All semantic networks
- `semantic_analysis_data['word_embeddings']` - Word vector representations
- `semantic_analysis_data['analysis_results']` - Complete analysis results
- `semantic_analysis_data['nlp_data']` - Processed linguistic data

**Your semantic word networks are ready for deep analysis and visualization! 🎉**

---